# HW5: plot your training logs
Download this notebook to your computer and open it in Jupyter or VS Code with a Python kernel. **This is the only course file you need for plotting:** the log reader, S3 downloader, and matplotlib plotting code are included below.

Install the packages in your notebook's Python environment once:
```sh
python -m pip install matplotlib boto3
```
For S3 downloads, configure AWS access on this computer in a terminal (`aws configure`, as in the handout). The notebook uses your configured credentials; do not put access keys in it. Local logs need no AWS access.

Run the cells from top to bottom. Edit only the download settings and plot settings.


## 1. Load the included helpers
Run this cell unchanged. No repository or separate Python helper file is needed.


In [ ]:
from pathlib import Path, PurePosixPath
import math
import re
import matplotlib.pyplot as plt

LOG_FILES = ("main.log", "run_parameters.txt", "args.json", "model_config.json", "samples.txt")


def parse_parameters(path):
    """Support legacy pipe tables and current key=value blocks; last block wins."""
    path = Path(path)
    values = {}
    if not path.exists():
        return values
    for line in path.read_text().splitlines():
        if line.strip().startswith("|"):
            parts = [p.strip() for p in line.split("|")]
            if len(parts) >= 4:
                values[parts[1]] = parts[2]
        elif not line.startswith("command:"):
            values.update(re.findall(r"(?:^|\s)([A-Za-z_]\w*)=([^\s]+)", line))
    for output, aliases, cast in (
        ("learning_rate", ("lr", "learning rate (LR)"), float),
        ("num_parameters", ("parameters", "num_parameters"), int),
    ):
        for alias in aliases:
            if alias in values:
                try:
                    values[output] = cast(values[alias])
                    break
                except ValueError:
                    continue
    return values


def read_log(path):
    """Deduplicate resumed steps. Preserve nonfinite observations as plot gaps."""
    streams, learning_rates, times = {}, {}, {}
    for number, line in enumerate(Path(path).read_text().splitlines(), 1):
        if not line.strip() or line.lstrip().startswith("#"):
            continue
        match = re.match(r"s:(\d+)\s+(trl|tel2|tel|eval):(\S+)", line)
        if not match:
            raise ValueError(f"{path}:{number}: not a training/validation metric record")
        step, stream, value = int(match[1]), match[2], float(match[3])
        streams.setdefault(stream, {})[step] = value if math.isfinite(value) else math.nan
        if stream == "trl":
            lr = re.search(r"\blr:(\S+)", line)
            if lr:
                learning_rates[step] = float(lr[1])
            timing = re.search(r"time\s*\(ms\):(\S+)", line)
            # A later record replaces the earlier timing for a repeated step.
            times.pop(step, None)
            if timing and math.isfinite(float(timing[1])):
                times[step] = float(timing[1])
    return {
        "streams": {name: sorted(rows.items()) for name, rows in streams.items()},
        "learning_rates": sorted(learning_rates.items()),
        "training_hours": sum(times.values()) / 3_600_000,
    }


def validation_label(pattern, fallback):
    parts = PurePosixPath(pattern).parts
    if "pretraining" in parts:
        return "pretraining-mix validation"
    for part in parts:
        if re.fullmatch(r"cluster_\d+", part):
            return f"{part} validation"
    if "expert" in parts:
        return "selected-cluster validation"
    return fallback


def plot_runs(run_dirs, output=None, loss_types=("trl", "tel", "tel2")):
    """Plot loss and actual scheduled LR; show HellaSwag only if it was logged."""
    import matplotlib.pyplot as plt

    runs = []
    for directory in run_dirs:
        directory = Path(directory)
        runs.append((directory, read_log(directory / "main.log"),
                     parse_parameters(directory / "run_parameters.txt")))
    if not runs:
        raise ValueError("Provide at least one run directory containing main.log")
    has_eval = any("eval" in data["streams"] for _, data, _ in runs)
    fig, axes = plt.subplots(1, 3 if has_eval else 2, figsize=(15 if has_eval else 11, 4.5))
    for index, (directory, data, params) in enumerate(runs):
        color = f"C{index % 10}"
        label = directory.name
        if isinstance(params.get("learning_rate"), (int, float)):
            label += f" (peak LR {params['learning_rate']:g})"
        primary = validation_label(params.get("validation", ""), "validation")
        secondary = validation_label(params.get("validation_secondary", ""), "second validation")
        for stream, style, title in (("trl", "-", "train"), ("tel", "--", primary), ("tel2", ":", secondary)):
            rows = data["streams"].get(stream, [])
            if rows and stream in loss_types:
                finite = [value for _, value in rows if math.isfinite(value)]
                minimum = f", min {min(finite):.4f}" if finite else ""
                axes[0].plot(*zip(*rows), style, color=color,
                             label=f"{label}: {title}{minimum}")
        if data["learning_rates"]:
            axes[1].plot(*zip(*data["learning_rates"]), color=color, label=label)
        if has_eval and data["streams"].get("eval"):
            axes[2].plot(*zip(*data["streams"]["eval"]), color=color, label=label)
    titles = [("Training and validation", "Loss (nats/token)"), ("Learning-rate schedule", "Actual base LR")]
    if has_eval:
        titles.append(("HellaSwag", "Accuracy"))
    for ax, (title, ylabel) in zip(axes, titles):
        ax.set(title=title, xlabel="Optimizer update", ylabel=ylabel)
        ax.grid(alpha=.25)
        if ax.lines:
            ax.legend(fontsize="small")
    fig.tight_layout()
    if output:
        output = Path(output)
        output.parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(output, dpi=180, bbox_inches="tight")
    return fig


def group_name(value):
    if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_.-]*", value) or value in (".", ".."):
        raise ValueError("Experiment group must be one name containing letters, digits, _, . or -")
    return value


def download_logs(client, bucket, group, destination):
    """Download logs only, even if the same S3 prefix also contains checkpoints."""
    prefix = f"experiments/{group_name(group)}/"
    destination = Path(destination)
    saved = []
    for page in client.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix):
        for item in page.get("Contents", []):
            key = item["Key"]
            if not key.startswith(prefix):
                continue
            relative = PurePosixPath(key[len(prefix):])
            if len(relative.parts) != 2 or relative.name not in LOG_FILES or any(p in (".", "..") for p in relative.parts):
                continue
            path = destination / group / relative.parts[0] / relative.name
            path.parent.mkdir(parents=True, exist_ok=True)
            client.download_file(bucket, key, str(path))
            saved.append(path)
    return saved


## 2. Download the logs
After uploading logs from EC2, set your bucket name and experiment group below. Part 1 uses `question_1_experiments`; Part 2 uses `full_gpt`; Part 3 uses `sft`.

This downloads only small log/settings files, not model checkpoints. If the logs are already on this computer, set `DOWNLOAD_FROM_S3 = False`.


In [ ]:
DOWNLOAD_FROM_S3 = True
BUCKET = "your_s3_bucket_name"
GROUP = "question_1_experiments"
LOG_DIR = Path("downloaded_logs") / GROUP

if DOWNLOAD_FROM_S3:
    import boto3
    if not BUCKET or BUCKET == "your_s3_bucket_name":
        raise ValueError("Set BUCKET to your S3 bucket name first.")
    files = download_logs(boto3.client("s3"), BUCKET, GROUP, Path("downloaded_logs"))
    downloaded_runs = sorted({p.parent.name for p in files if p.name == "main.log"})
    if not downloaded_runs:
        raise ValueError("No main.log files found. Check BUCKET, GROUP, and your upload.")
    print("Downloaded runs:", ", ".join(downloaded_runs))
    print("Logs saved in:", LOG_DIR.resolve())


## 3. Choose the plot
- `RUN_DIRS`: keep one directory for an individual experiment, or both to compare them. You can also use any local run directory containing `main.log`.
- `LOSS_TYPES`: use `["trl", "tel", "tel2"]` for training and validation, `["trl"]` for training only, or `["tel", "tel2"]` for validation only. Missing curves are simply omitted. The legend reports each curve's minimum loss.
- `OUTPUT`: choose a different PNG filename for each plot you want to keep.

**Part 2:** download group `full_gpt`, then set `RUN_DIRS = [LOG_DIR / "expert"]` and `OUTPUT = "plots/expert.png"`. Keep all three loss types: training, selected-cluster validation (`tel`), and pretraining-mix validation (`tel2`). Both validation curves come from the supplied HF validation files.

**Part 3 (SFT):** download group `sft`, then set `RUN_DIRS = [LOG_DIR / "sft"]`, `LOSS_TYPES = ["trl", "tel"]`, and `OUTPUT = "plots/sft.png"`. This plots training and held-out conversation validation loss. SFT starts its own update counter; its loss is not directly comparable with pretraining loss.

Use the same validation dataset and evaluation budget when comparing candidate learning rates.


In [ ]:
RUN_DIRS = [LOG_DIR / "full_run", LOG_DIR / "question_1_experiment_2"]
LOSS_TYPES = ["trl", "tel", "tel2"]
OUTPUT = "plots/question_1.png"


In [ ]:
runs = [Path(p) for p in RUN_DIRS]
for run in runs:
    if not (run / "main.log").is_file():
        raise FileNotFoundError(f"{run}/main.log not found. Set RUN_DIRS to your log directories.")
    data = read_log(run / "main.log")
    params = parse_parameters(run / "run_parameters.txt")
    vals = data["streams"].get("tel", [])
    print(run.name, "peak LR:", params.get("learning_rate", "unknown"),
          "parameters:", params.get("num_parameters", "unknown"),
          "last validation:", vals[-1] if vals else "not logged",
          "logged training hours:", round(data["training_hours"], 3))
fig = plot_runs(runs, OUTPUT, loss_types=LOSS_TYPES)
plt.show()
print("Plot saved to:", Path(OUTPUT).resolve())


The loss panel uses solid lines for training, dashed lines for primary validation, and dotted lines for the second validation set. The next panel shows the actual learning-rate schedule; HellaSwag appears only when logged.

Resumed runs keep absolute optimizer-update numbers. Repeated steps use the latest record, and nonfinite values appear as gaps. Logged training time excludes loading, validation, and other overhead; use wall-clock time for cost estimates. SFT logs work too, but SFT starts its own optimizer and step counter.
